# 1. Composition

## 1.2 Getting values according Eskici and Axelsen equation

\begin{equation}
n_{\text{AOT}}=36\pi\left(W_{0}+\cfrac{3}{2}\right)^{2}
\cfrac{\bar{v}^{2}}{\left(a_{\text{AOT}}\right)^{3}}
   +n_{\text{AOT}}^{0}
\label{ec:eskici_01} 
\end{equation}

In [1]:
import numpy as np

def eskici_naot(**kwargs):
    
    # default values in paper, constants
    no_aot = kwargs.get("no_aot", 13.6) # adimensional
    a_aot = kwargs.get("a_aot", 59) # (angstroms)^2
    vbar = kwargs.get("vbar", 33) # (angstroms)^3
    
    w0 = kwargs.get("w0", 7.5) # independent value
    
    A = np.divide(np.power(vbar, 2), np.power(a_aot, 3))
    B = 36*np.pi*np.power((w0 + 1.5), 2)
    
    n_aot = A*B + no_aot
    
    return n_aot
    
    
w0=np.array([0, 2.5, 5, 7.5, 10, 12.5, 15.0])
eskici_naot(w0=w0)
np.round(eskici_naot(w0=w0))


# rms_meoh

array([ 15.,  23.,  39.,  62.,  93., 131., 177.])

The `eskici_naot` method is added to the `rms_meoh` class and is used in the following way, along with the `waters4Wos` method, which specifies the number of water molecules per charge required

# 1.3 Number of AOT and Water molecules for water loads from 0 to 15

In [2]:
import sys 
sys.path.append('../code/')
from rms_meoh import *

rm = rms_meoh()

# in the paper we only use values from 0 to 7.5
w0s = np.array([0, 2.5, 5, 7.5, 10, 12.5, 15.0])

for w0p in w0s:

    w0 = w0p
    num_aot = int(np.round(rm.eskici_naot(w0=w0)))
    num_h2o = int(np.round(rm.waters4Wos(w0=w0, aots=num_aot)))
    
    print(f"w0: {w0}, aots: {num_aot}, h20: {num_h2o}")
    
    

w0: 0.0, aots: 15, h20: 0
w0: 2.5, aots: 23, h20: 58
w0: 5.0, aots: 39, h20: 195
w0: 7.5, aots: 62, h20: 465
w0: 10.0, aots: 93, h20: 930
w0: 12.5, aots: 131, h20: 1638
w0: 15.0, aots: 177, h20: 2655


For our research, we used only water loads of $w_{0}$= [0, 2.5, 5, 7.5]

# 2.1 Molecular Dynamics of the Self-Assembled RMs

To achieve this, we have created a script that streamlines the setup and simulation of the system for any composition

In [1]:
import sys 
sys.path.append('../code/')
from rms_meoh import *
from norm import *
import getpass
from herramientas import *

user = getpass.getuser()

rm = rms_meoh()
nr = norm()
herr = herramientas()

gmx = '/usr/local/gromacs/bin/gmx' # leave it 
gmx = "/home/antadlp/gmx_2021_7/bin/gmx"
NUM_PROC = 6

PATH_GRL_0_0 = "/home/{}/Documents/cajas_norm_ws_01".format(user)
NAME_CARPET0 = "test_create_rms_001"

PATH_GRO_NA = "../data/gros/na.gro"
PATH_GRO_AOT  = "../data/gros/aot.gro"
PATH_GRO_ISO = "../data/gros/isooctano.gro"
PATH_GRO_H2O = "../data/gros/spc.gro"

PATH_FF_AOT = "../data/itps/63UD_GROMACS_G54A7FF_allatom_UDD.itp"
PATH_FF_H20 = "../data/itps/spc_54a7.itp"
PATH_FF_NA  = "../data/itps/E0XM_GROMACS_G54A7FF_allatom_original.itp"
PATH_FF_ISO = "../data/itps/G016_ffbonded_zero_charge.itp"
PATH_FF_GROMOS54A7_ATB = "../data/itps/ffnonbonded_gromos54a7_atb_original.itp"

# nsteps_min1 = 10000
# nsteps_min2 = 10000
# nsteps_nvt1 = 10000000

nsteps_min1 = 50000
nsteps_min2 = 50000
nsteps_nvt1 = 100000

dt_nvt1=0.001

rcoulomb_nvt1=1.0
rvdw_nvt1=1.0
temp=298.15
# nstout=2000
# nstlog=2000

nstout=500
nstlog=500


fiso=0.35
w0s = np.array([0, 2.5, 5, 7.5, 10, 12.5, 15.0])
Ls = [7, 7, 10, 11, 12, 13, 14]
d = 1.5

w0s = np.array([0, 2.5, 5, 7.5, 10])
Ls = [7, 7, 10, 11, 12]

for w0p, L in zip(w0s, Ls):

    w0 = w0p
    num_aot = int(np.round(rm.eskici_naot(w0=w0)))
    num_h2o = int(np.round(rm.waters4Wos(w0=w0, aots=num_aot)))
    _ = rm.f_Niso(fiso=fiso, Nhoh=num_h2o, Naot=num_aot, Nna=num_aot)
    num_iso = int(np.round(_)) # only in the case you want some isooctane in the self-assembly process
    num_iso = 0 # only RMs

    print("w0: {}, aots: {}, h20: {}, iso: {}".format(w0, num_aot, num_h2o, num_iso))

    NAME_DIN = "R_" + str(num_aot) + "_" + str(num_h2o)

    dP_LVL0 = nr.set_paths_lvl0(path_grl_0_0=PATH_GRL_0_0,
                                name_carpet0=NAME_CARPET0,
                                path_gro_aot=PATH_GRO_AOT,
                                path_gro_iso=PATH_GRO_ISO,
                                path_gro_h2o=PATH_GRO_H2O,
                                path_gro_na=PATH_GRO_NA,
                                path_ff_aot=PATH_FF_AOT,
                                path_ff_h2o=PATH_FF_H20,
                                path_ff_na=PATH_FF_NA,
                                path_ff_iso=PATH_FF_ISO,
                                path_ff_54a7=PATH_FF_GROMOS54A7_ATB)

    dP_LVL1 = nr.set_paths_lvl1(name_din=NAME_DIN, dP_LVL0=dP_LVL0)

    PATH_CELDA0 = nr.create_no_rm(gmx=gmx,
                               num_aot=num_aot,
                               num_h2o=num_h2o,
                               num_iso=num_iso,
                               L=L,
                               dP_LVL0=dP_LVL0,
                               dP_LVL1=dP_LVL1)
    
    
    nr.get_ffnb_no_rm(carpet="min1",
                      dP_LVL0=dP_LVL0,
                      dP_LVL1=dP_LVL1,
                      epsilon_factor=0.925)
    
    
    nr.create_topol(carpet="min1",
                    dP_LVL0=dP_LVL0,
                    dP_LVL1=dP_LVL1)
    
    
    to = time.time()
    nr.do_min1(gmx=gmx,
               num_proc=NUM_PROC,
               nsteps=nsteps_min1,
               dP_LVL1=dP_LVL1)
    t = time.time() - to
    print("{}: min1: {}".format(w0, t))
    
    to = time.time()
    nr.do_min2(gmx=gmx,
               num_proc=NUM_PROC,
               nsteps=nsteps_min1,
               dP_LVL1=dP_LVL1)
    t = time.time() - to
    print("{}: min2: {}".format(w0, t))  
    
    

    
    to = time.time()
    nr.wrapper_nvt1(dP_LVL1=dP_LVL1,
                     num_proc=NUM_PROC,
                     gmx=gmx,
                     nsteps=nsteps_nvt1,
                     dt=dt_nvt1,
                     rcoulomb=rcoulomb_nvt1,
                     rvdw=rvdw_nvt1,
                     nstout=nstout,
                     nstlog=nstlog,
                     temp=temp)
    t = time.time() - to
    print("{}: nvt1: {}".format(w0, t))  
    
    print("w0: {} DONE".format(w0))





w0: 0.0, aots: 15, h20: 0, iso: 0


                 :-) GROMACS - gmx insert-molecules, 2021.7 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            Tee

0.0: min1: 52.927870988845825


Using 1 MPI thread
Using 6 OpenMP threads 


NOTE: The number of threads is not equal to the number of (logical) cores
      and the -pin option is set to auto: will not pin threads to cores.
      This can lead to significant performance degradation.
      Consider using -pin on (and -pinoffset in case you run multiple jobs).

Steepest Descents:
   Tolerance (Fmax)   =  1.00000e+01
   Number of steps    =        50000

Energy minimization has stopped, but the forces have not converged to the
requested precision Fmax < 10 (which may not be possible for your system). It
stopped because the algorithm tried to make a new step whose size was too
small, or there was no change in the energy since last step. Either way, we
regard the minimization as converged to within the available machine
precision, given your starting configuration and EM parameters.

Double precision normally gives you higher accuracy, but this is often not
needed for preparing to run molecular dynamics.

writing lowest e

0.0: min2: 5.460321664810181


...

Using 1 MPI thread
Using 6 OpenMP threads 


NOTE: The number of threads is not equal to the number of (logical) cores
      and the -pin option is set to auto: will not pin threads to cores.
      This can lead to significant performance degradation.
      Consider using -pin on (and -pinoffset in case you run multiple jobs).

Steepest Descents:
   Tolerance (Fmax)   =  1.00000e+01
   Number of steps    =        50000

Energy minimization has stopped, but the forces have not converged to the
requested precision Fmax < 10 (which may not be possible for your system). It
stopped because the algorithm tried to make a new step whose size was too
small, or there was no change in the energy since last step. Either way, we
regard the minimization as converged to within the available machine
precision, given your starting configuration and EM parameters.

Double precision normally gives you higher accuracy, but this is often not
needed for preparing to run molecular dynamics.

writing lowest e

10.0: min2: 221.14917278289795


Changing nstlist from 10 to 100, rlist from 1.016 to 1.239

1 GPU selected for this run.
Mapping of GPU IDs to the 1 GPU task in the 1 rank on this node:
  PP:0
PP tasks will do (non-perturbed) short-ranged interactions on the GPU
PP task will update and constrain coordinates on the CPU
Using 1 MPI thread
Using 6 OpenMP threads 


NOTE: The number of threads is not equal to the number of (logical) cores
      and the -pin option is set to auto: will not pin threads to cores.
      This can lead to significant performance degradation.
      Consider using -pin on (and -pinoffset in case you run multiple jobs).
starting mdrun 'R_93_930'
100000 steps,    100.0 ps.


10.0: nvt1: 36.57470369338989
w0: 10.0 DONE



Writing final coordinates.

               Core t (s)   Wall t (s)        (%)
       Time:      217.663       36.277      600.0
                 (ns/day)    (hour/ns)
Performance:      238.167        0.101

GROMACS reminds you: "Sacrifices must be made" (Otto Lilienthal, dying after having crashed with his glider in 1896)



In the first set of images, the initial configurations are shown, while in the second set, the molecules start to cluster together. Some fully formed clusters can already be observed.The configurations shown correspond to the previous cell and are not from the article. They are examples to demonstrate how the script that generates the RMs works. As stated in the article, at least 20ns of simulation is required for the formation of RMs.


In [ ]:
from IPython.display import Image
# Image(filename="../imgs/celda0_RM_62.png")

The next step, assuming the clusters have already formed, is to center the cluster in the middle of the cell. This is necessary because, as was the case with the RMs used in the article, they tended to remain stuck to the cell walls during simulation, rendering them unusable without additional processing. This problem was caused by periodic boundary conditions. The solution to this problem was discovered after extensive time and numerous attempts with various tools (including k-means clustering). It involves using the following GROMACS instructions.

In [3]:
# gmx trjconv -f ../traj.xtc -skip 10 -o traj10skip.xtc
# gmx trjconv -f traj10skip.xtc -o pbcRes.gro -pbc res
# gmx trjconv -f pbcRes.gro -o pbcResCluster.gro -pbc cluster
# gmx trjconv -f pbcResCluster.gro -o pbcResClusterC.gro -center 

The simulation results for 23 and 39 AOTs used in the published article can be downloaded. These can be used to test the reconstruction of the AOT aggregate due to the problem of boundary conditions